# Marker volcano plots

Volcano plots of per-gene knockdown effects on marker-derived image features. For each
marker/feature pair, every targeted gene's median feature value is compared to the
non-targeting-control (NTC) median; the x axis is that effect size and the y axis is the
FDR-corrected significance of the corresponding permutation test.

Input data is the per-marker statistics table produced by the `paper_v2` marker volcano
analysis, subset to the curated (marker, feature) combinations for the figure and copied
into `../../data/figures/figure_1/` by `data_preprocessing/figure_1.py`.


## Imports


In [ ]:
import contextlib
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from adjustText import adjust_text

## Input path and plotting parameters

`marker_volcano_stats.csv` holds one row per (marker, feature, gene) with the gene median,
the NTC median, the effect relative to NTC, and the raw/FDR-corrected p-values. It contains
only the curated marker/feature combinations, in the panel order set by
`MARKER_VOLCANO_PANELS` in `data_preprocessing/figure_1.py`.


In [ ]:
VOLCANO_STATS_CSV = Path("../../data/figures/figure_1/marker_volcano_stats.csv")
OUTPUT_DIR = Path("../../output/figure_1")

FDR_THRESHOLD = 0.05
N_LABEL = 10  # number of significant genes to label per panel, by |effect size|

COLORS = {
    "ns": "#d9d9d9",
    "up": "#c0392b",
    "down": "#2c6fbb",
    "ntc": "#f0a030",
}

## Load statistics table


In [ ]:
stats_df = pd.read_csv(VOLCANO_STATS_CSV)

# Panel order comes from the file, which the preprocessing script writes in curated order.
panels = stats_df[["marker", "feature"]].drop_duplicates().reset_index(drop=True)

print(f"Loaded {len(stats_df)} rows: {stats_df['marker'].nunique()} markers, {len(panels)} marker/feature panels")
print(panels.to_string())
print(f"Genes per panel: {stats_df.groupby(['marker', 'feature']).size().unique()}")
print(f"NTCs per panel  : {stats_df.groupby(['marker', 'feature'])['is_ntc'].sum().unique()}")

stats_df.head()

## Significant hits per panel

Number of targeted genes passing the FDR threshold in each marker/feature panel, split by
direction of effect, in curated panel order.


In [ ]:
hit_counts = (
    stats_df[stats_df["significant"] & ~stats_df["is_ntc"]]
    .groupby(["marker", "feature", "direction"])
    .size()
    .unstack("direction", fill_value=0)
    .reindex(pd.MultiIndex.from_frame(panels))
    .fillna(0)
    .astype(int)
)
hit_counts["total"] = hit_counts.sum(axis=1)
hit_counts

## Volcano panel

One panel per marker/feature pair. Non-significant genes are grey, significant genes are
coloured by direction of effect, and NTC guides are drawn in orange to show the null
spread around zero effect. The dashed line marks the FDR threshold and the top `N_LABEL`
significant genes by absolute effect size are labelled.


In [ ]:
def plot_volcano(
    ax: plt.Axes,
    panel_df: pd.DataFrame,
    n_label: int = N_LABEL,
    fontsize: float = 7,
    label_fontsize: float = 6,
) -> None:
    """Draw a single marker/feature volcano panel onto `ax`."""
    marker = panel_df["marker"].iloc[0]
    feature = panel_df["feature"].iloc[0]

    genes = panel_df[~panel_df["is_ntc"]]
    ntcs = panel_df[panel_df["is_ntc"]]

    ns = genes[~genes["significant"]]
    ax.scatter(ns["effect_vs_ntc"], ns["neglog10_qval"], s=8, c=COLORS["ns"], linewidths=0, label="n.s.")

    for direction in ("down", "up"):
        sig = genes[genes["significant"] & (genes["direction"] == direction)]
        ax.scatter(
            sig["effect_vs_ntc"],
            sig["neglog10_qval"],
            s=12,
            c=COLORS[direction],
            linewidths=0,
            label=f"{direction} (n={len(sig)})",
        )

    ax.scatter(
        ntcs["effect_vs_ntc"],
        ntcs["neglog10_qval"],
        s=10,
        c=COLORS["ntc"],
        linewidths=0,
        label=f"NTC (n={len(ntcs)})",
    )

    ax.axhline(-np.log10(FDR_THRESHOLD), color="grey", linestyle="--", linewidth=0.8)
    ax.axvline(0.0, color="grey", linestyle=":", linewidth=0.8)

    # Many significant genes share the permutation-floor q-value, so leave vertical
    # headroom above the top row of points for the labels to be pushed into.
    ax.set_ylim(top=panel_df["neglog10_qval"].max() * 1.35)

    sig_genes = genes[genes["significant"]]
    top = sig_genes.reindex(sig_genes["effect_vs_ntc"].abs().sort_values(ascending=False).index).head(n_label)
    texts = [
        ax.text(row.effect_vs_ntc, row.neglog10_qval, row.gene_symbol, fontsize=label_fontsize)
        for row in top.itertuples()
    ]
    if texts:
        # adjustText 1.3.0 prints jitter debug lines when labels share coordinates
        # (common here: many hits sit at the permutation-floor q-value).
        with contextlib.redirect_stdout(io.StringIO()):
            adjust_text(
                texts,
                ax=ax,
                expand=(1.3, 1.6),
                force_text=(0.4, 0.8),
                arrowprops=dict(arrowstyle="-", color="grey", lw=0.4),
            )

    ax.set_title(f"{marker}\n{feature}", fontsize=fontsize)
    ax.set_xlabel("median effect vs NTC", fontsize=fontsize)
    ax.set_ylabel("-log10 FDR", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize - 1)

## All panels

Every curated marker/feature panel on a shared grid, for a quick overview of which markers
show strong knockdown phenotypes.


In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(panels) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.2 * n_rows))
axes_flat = axes.ravel()

for ax, (marker, feature) in zip(axes_flat, panels.itertuples(index=False, name=None)):
    panel_df = stats_df[(stats_df["marker"] == marker) & (stats_df["feature"] == feature)]
    plot_volcano(ax, panel_df, n_label=5)

for ax in axes_flat[len(panels):]:
    ax.set_visible(False)

fig.tight_layout()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "volcano_plots_all_markers.pdf", dpi=150, bbox_inches="tight")
plt.show()

## Individual panels

Full-size volcano plot per marker/feature pair, written to
`../../output/figure_1/volcano_plots/<marker>__<feature>.pdf`.


In [ ]:
panel_dir = OUTPUT_DIR / "volcano_plots"
panel_dir.mkdir(parents=True, exist_ok=True)

for marker, feature in panels.itertuples(index=False, name=None):
    panel_df = stats_df[(stats_df["marker"] == marker) & (stats_df["feature"] == feature)]

    fig, ax = plt.subplots(figsize=(6, 5))
    plot_volcano(ax, panel_df, fontsize=10, label_fontsize=8)
    ax.legend(fontsize=7, frameon=False, loc="best")
    fig.tight_layout()
    fig.savefig(panel_dir / f"{marker}__{feature}.pdf", dpi=150, bbox_inches="tight")
    plt.show()

print(f"Wrote {len(panels)} volcano panels to {panel_dir}")